# Here the time performance of the code is generated

In [ ]:
import re
import pickle
import os
import sys
import numpy as np
import matplotlib.pyplot as plt
import function_definitions as field_funcs
import OCP_construction_functions as OCP_funcs
import sympy
import helper_funcs as hfct
import scipy
import scipy.optimize as opt


def get_floats_from_filename(filename):
    return re.findall(r"[+-]? *(?:\d+(?:\.\d*)?|\.\d+)(?:[eE][+-]?\d+)?", filename)


def create_dict_for_folder(folderdir):
    file_names_in_folder = os.listdir(folderdir)
    stepsize_filename_dict = dict()
    for x in file_names_in_folder:
        stepsize_filename_dict[x] = [float(y)  for y in get_floats_from_filename(x)] #assume fix order a,b,gamma,h
    return stepsize_filename_dict  

base_folder_dir = 'data/low_thrust/polar_variables'
def get_data_from_folder(foldername):
    folder_names = os.listdir(foldername+'/')
    dict_of_folders_with_files = dict()
    for x in folder_names:
        if x != '.DS_Store':
            dict_of_folders_with_files[x] = create_dict_for_folder(foldername+'/' +x)
    #Structure of data dict:
        # data_dict["folder_name"]=[ {"file_name":data}, [alpha,beta,gamma,h] ]
    data_dict = dict()
    for x in dict_of_folders_with_files.keys():
        for y in dict_of_folders_with_files[x].keys():
            if y == '.DS_Store':
                continue

            filename = foldername+"/"+x + "/"+y
            with open(filename, 'rb') as files:
                data_dict[y] = [pickle.load(files) ,dict_of_folders_with_files[x][y]]
    return data_dict            

base_folder_dir_test = base_folder_dir
base_foldername = base_folder_dir_test 
data=get_data_from_folder(base_foldername)


In [ ]:

file_name_to_use = 'data_a=0.5g=1N=100.pkl'
data_to_use = data[file_name_to_use][0]

def compute_average_computation_time(data_to_use):
    parameters_to_use = data_to_use['parameters']
    parameters_to_use['times'] = sympy.Matrix(parameters_to_use['times'])
    parameters_to_use['q0'] = sympy.Matrix(parameters_to_use['q0'])
    parameters_to_use['dq0'] = sympy.Matrix(parameters_to_use['dq0'])
    parameters_to_use['dqT'] = sympy.Matrix(parameters_to_use['dqT'])
    parameters_to_use['dq0'] = sympy.Matrix(parameters_to_use['dq0'])



    continuous_eq = OCP_funcs.Direct_continuous_generator(parameters_to_use,f_func=field_funcs.f_vec_polar,rho_func=field_funcs.rho_vec_polar,running_cost_func=field_funcs.running_cost_polar,mayer_func=field_funcs.mayer_term_polar,g_func=field_funcs.g_mat_polar)

    discrete_equations = OCP_funcs.discrete_standard_direct_eq_generator(continuous_eq)

    cs_u= scipy.interpolate.CubicSpline(np.linspace(0,parameters_to_use["T"],len(data_to_use['u_d_new'])),data_to_use['u_d_new'])

    U_d_1_use = cs_u(np.array(parameters_to_use["times"]) + parameters_to_use["gamma"]*parameters_to_use["h"])
    U_d_2_use =cs_u(np.array(parameters_to_use["times"]) + (1-parameters_to_use["gamma"])*parameters_to_use["h"])

    q_d_use = data_to_use['q_d_new']
    lambda_d_use = data_to_use['lambda_d_new']
    lambda_q_d_use = data_to_use['lambda_q_standard']
    lambda_v_d_use = data_to_use['lambda_v_standard']
    mu_use = [0.1,0.1]
    nu_use = [0.1,0.1]
    v_q_use = data_to_use['v_y_d_new'][:,:2]

    eval_arg_standard = mu_use + nu_use + list(q_d_use.flatten()) + list(v_q_use.flatten()) + list(lambda_q_d_use.flatten()) + list(lambda_v_d_use.flatten()) +list(U_d_1_use.flatten())+list(U_d_2_use.flatten())
    eval_arg_new = mu_use + nu_use + list(q_d_use.flatten()) + list(lambda_d_use.flatten()) +list(U_d_1_use.flatten())+list(U_d_2_use.flatten())
    eval_arg_new_no_u = mu_use + nu_use + list(q_d_use.flatten()) + list(lambda_d_use.flatten()) 


    # Here create the lambdified stuff
    #standard
    standard_direct_midpoint_KKT = discrete_equations.calc_KKT()
    lambdified_KKT =  sympy.lambdify(standard_direct_midpoint_KKT[1],standard_direct_midpoint_KKT[0]) 
    lambdified_KKT_eval = lambda x :lambdified_KKT(*x)
    # new
    standard_direct_KKT_new = discrete_equations.calc_KKT_new()
    lambdified_KKT_new =  sympy.lambdify(standard_direct_KKT_new[1],standard_direct_KKT_new[0])
    lambdified_KKT_new_eval = lambda x :lambdified_KKT_new(*x)
    # new no u
    standard_direct_KKT_new_no_u = discrete_equations.calc_KKT_new_no_u()
    lambdified_KKT_new_no_u =  sympy.lambdify(standard_direct_KKT_new_no_u[1],standard_direct_KKT_new_no_u[0])
    lambdified_KKT_new_eval_no_u = lambda x :lambdified_KKT_new_no_u(*x)


    result_standard = %timeit -o lambdified_KKT_eval(eval_arg_standard)
    mean_standard =np.mean(result_standard.all_runs)/result_standard.loops
    std_standard = result_standard.stdev
    result_new = %timeit -o lambdified_KKT_new_eval(eval_arg_new)
    mean_new =np.mean(result_new.all_runs)/result_new.loops
    std_new = result_new.stdev

    result_new_no_u = %timeit -o lambdified_KKT_new_eval_no_u(eval_arg_new_no_u)
    mean_new_no_u =np.mean(result_new_no_u.all_runs)/result_new_no_u.loops
    std_new_no_u = result_new_no_u.stdev

    return {'standard':[mean_standard,std_standard], 'new':[mean_new,std_new], 'new no u': [mean_new_no_u,std_new_no_u]}



In [ ]:
computation_time_data = dict()
for tmp in data.keys():
    if 'ref' in tmp:
        continue
    tmp_data = data[tmp][0]
    print('currently working on ' + tmp)
    time_result = compute_average_computation_time(tmp_data)
    computation_time_data[tmp] = time_result


from pathlib import Path 
comp_time_save_file_name =base_folder_dir + '/function_computation_time'

Path(comp_time_save_file_name).mkdir(parents=True, exist_ok=True)

file_name = comp_time_save_file_name+"/time_comparison_data.pkl"
with open(file_name, 'wb') as ffile:
    pickle.dump(computation_time_data, ffile)
    ffile.close()